In [1]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments
from tqdm.notebook import tqdm
import pandas as pd
from datasets import Dataset, load_metric, DatasetDict
model_name = 'facebook/bart-base'
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("device ",device)
model = model.to(device)

2024-08-20 20:51:39.752738: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-20 20:51:39.789186: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-20 20:51:40.539317: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(

device  cuda


In [2]:
mapping = {"N":"negation", "E":"entity swap", "#":"number swap", 
           "A":"to antonym", "I":"no information", 
           "X":"mutual exclusion"}


In [3]:

def mission(context, question, sal):
    st = f"""Context: {context}

    Question: {question}

    Salient Sentence: {sal}

    Task: Generate a new question based on the context and salient sentence that can be answered with the given context."""
     
    return st

def tokenize_function(examples):
    # Prepare inputs
    inputs = [
        mission(context if label != "I" else noinfoc, question if label != "I" else noinfoq, sal)
        for context, question, label, sal, noinfoc, noinfoq in zip(examples['context'], examples['potential_UQ_question'], examples['label'], examples['answer_sentence'], examples['old_context'], examples['original_question'])
    ]
    targets = [
        original_q if label != "I" else potential_q 
        for label, original_q, potential_q in zip(examples['label'], examples['original_question'], examples['potential_UQ_question'])
    ]
    
    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')
    
    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding='max_length')
    
    # Ensure the labels are correctly formatted
    labels_ids = labels['input_ids']
    # Align the lengths of labels and inputs

    model_inputs['labels'] = labels_ids

    return model_inputs

In [4]:
csv_file = 'all_unans_salience.csv'  # Replace with your CSV file path
df = pd.read_csv(csv_file)
#df = df.loc[df['label'] != 'Ans']
df

,Unnamed: 0,context,original_question,potential_UQ_question,original_answer_span,answer_sentence,label,old_context,label_indicator
0,0,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
1,1,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
2,2,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
3,3,"In October 2015, TCM announced the launch of t...",How long do TCM Wineclub subscriptions last?,How unretentive do TCM Wineclub subscriptions ...,3 month,"Wines are available in 3 month subscriptions, ...",A,NaN,0
4,4,Some Christian writers considered the possibil...,As what was the event mistaken by some pagans?,As what was the event mistaken by no pagans?,a solar eclipse,Some Christian writers considered the possibil...,A,NaN,0
...,...,...,...,...,...,...,...,...,...
20471,20471,Most residents belong to the Anglican Communio...,The Diocese of Saint Helena has it's own what?,The Diocese of Saint Helena has it's own what?,bishop,Most residents belong to the Anglican Communio...,Ans,NaN,1
20472,20472,Other Christian denominations on the island in...,What year did the Salvation Army show up on Sa...,What year did the Salvation Army show up on Sa...,1884,Other Christian denominations on the island in...,Ans,NaN,1
20473,20473,Other Christian denominations on the island in...,What year did the Salvation Army show up on Sa...,What year did the Salvation Army show up on Sa...,1884,Other Christian denominations on the island in...,Ans,NaN,1
20474,20474,Other Christian denominations on the island in...,When did Baptists come to the island?,When did Baptists come to the island?,1845,Other Christian denominations on the island in...,Ans,NaN,1


In [5]:
# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
# Split the dataset into train and validation sets
dataset = dataset.train_test_split(test_size=0.2)
dataset = DatasetDict({"train": dataset["train"], "validation": dataset["test"]})
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16380 [00:00<?, ? examples/s]

/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

In [6]:
def filter_dict(example):
    return {
        'input_ids': example['input_ids'],
        'attention_mask': example['attention_mask'],
        'labels': example['labels']
    }
# Apply the filter function to each split
filtered_dataset_dict = {}
for split, dataset in tokenized_datasets.items():
    filtered_dataset_dict[split] = dataset.map(
        filter_dict, 
        remove_columns=[col for col in dataset.column_names if col not in ['input_ids', 'attention_mask', 'labels']]
    )

Map:   0%|          | 0/16380 [00:00<?, ? examples/s]

Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

In [7]:
train = filtered_dataset_dict['train']
validation = filtered_dataset_dict['validation']

In [8]:
def compute_metrics(eval_pred):
    metric = load_metric("accuracy")
    logits, labels = eval_pred
    predictions = torch.argmax(logits, dim=-1)
    return metric.compute(predictions=predictions, references=labels)

In [9]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir='./results',  # Specify where to save checkpoints and outputs
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    learning_rate=5e-5,
    save_steps=500,            # Save a checkpoint every 500 steps
    save_total_limit=2,
    fp16=True# Keep only the last 2 checkpoints
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=validation,
    #compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train the model
#trainer.train()
#checkpoint_path = "./results/checkpoint-11000"  # Example checkpoint path
trainer.train()#resume_from_checkpoint=checkpoint_path)
# Save the model


/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: julien-serbanescu (juteam). Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
0,0.036800,0.035350
2,0.013100,0.033916
4,0.007800,0.036704
5,0.003400,0.037828


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file

TrainOutput(global_step=12282, training_loss=0.04565083205205066, metrics={'train_runtime': 2264.0492, 'train_samples_per_second': 43.409, 'train_steps_per_second': 5.425, 'total_flos': 2.995513272041472e+16, 'train_loss': 0.04565083205205066, 'epoch': 5.9985347985347985})

In [10]:
model.save_pretrained('./fine-tuned-bart-salience')
tokenizer.save_pretrained('./fine-tuned-bart-salience')

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


('./fine-tuned-bart-salience/tokenizer_config.json',
 './fine-tuned-bart-salience/special_tokens_map.json',
 './fine-tuned-bart-salience/vocab.json',
 './fine-tuned-bart-salience/merges.txt',
 './fine-tuned-bart-salience/added_tokens.json')